# 18 — Bài báo, bước 3a: prompt trung tính về ứng dụng cho LLM (máy local, API)

**Câu hỏi**

Mức giảm khoảng 20 điểm Macro-F1 của LLM từ Hold-out Shopee sang Lazada có do prompt nêu đích danh Shopee hay không.

**Biến thể**

Ba prompt v1 đã khóa (tầng 1, tầng 2, phẳng) chỉ bỏ chữ "Shopee" khỏi câu mở đầu (bản `v1n`); quy tắc, ví dụ few-shot, model,
temperature, seed API và schema giữ nguyên. Thiết kế đã pre-register ở amendment v3.6; biến thể chạy một lần, không chỉnh trên Lazada.

**Kiểm soát chi phí**

Chạy thử 20 review Lazada text-only trước. Lazada (`RUN_LAZADA`) và Hold-out tùy chọn (`RUN_HOLDOUT`) chỉ chạy khi người nghiên cứu
đã xác nhận; prompt `v1n` được khóa vào registry trước lần chạy đầy đủ đầu tiên. Log không chứa văn bản review; chạy lại thì đọc từ log.

In [1]:
RUN_LAZADA = True     # chỉ bật sau khi người nghiên cứu xác nhận chi phí
RUN_HOLDOUT = False    # tùy chọn, cũng cần xác nhận

import os, sys, json, datetime, subprocess
import numpy as np, pandas as pd
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification.evaluate import FOUR_CLASSES
from src.classification import paper_analysis as PA
from src.classification.final_eval import llm_ids
from src.classification.strategy_llm_fewshot import prompts as P
from src.classification.strategy_llm_fewshot.classify import classify, cost_summary, results_from_log
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
DATA = os.path.join(ROOT, '01_data', 'processed')
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
RUN_DATE = datetime.date.today().strftime('%d/%m/%Y')
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
train_df = pd.read_csv(os.path.join(DATA, 'shopee_train.csv'), encoding='utf-8-sig')
TEXT = {s: pd.read_csv(os.path.join(DATA, f'{s}_text_only.csv'), encoding='utf-8-sig') for s in ('lazada', 'holdout')}
for s, d in TEXT.items():
    assert list(d.columns) == ['review_id', 'text']
print('Ngày chạy:', RUN_DATE, '| commit:', COMMIT, '| RUN_LAZADA:', RUN_LAZADA, '| RUN_HOLDOUT:', RUN_HOLDOUT)

Ngày chạy: 15/11/2026 | commit: 9e19ef75c14d7dd0ebc9aa95cda542a8238a303d | RUN_LAZADA: True | RUN_HOLDOUT: False


## Kiểm tra API

**Nội dung**

Một lần gọi ngắn với snapshot đã khóa; model trả về phải trùng snapshot. Key đọc từ `.env` và không được in ra.

In [2]:
from src.utils.llm import check_api, get_client, prices
r = check_api()
assert r['ok'], f"API lỗi: {r.get('error')} — {r.get('hint', '')}"
assert r['snapshot_match'], 'Model trả về khác snapshot đã khóa'
PRICES = prices()
client, SETTINGS = get_client()
print('Model:', r['response_model'], '| giá / 1M token:', PRICES)

Model: gpt-4o-mini-2024-07-18 | giá / 1M token: {'input': 0.15, 'output': 0.6, 'source': 'env'}


## Prompt trung tính

**Tạo bản v1n**

`write_neutral` đổi đúng một cụm trong system prompt của mỗi bản v1 và chép nguyên manifest ví dụ. Hash ba prompt v1 phải vẫn khớp
registry; hash bản v1n được in ra và chỉ được khóa trước lần chạy đầy đủ.

In [3]:
REG = open(P.REGISTRY, encoding='utf-8').read()
TIERS = {'tier1': 1, 'tier2': 2, 'flat': 'flat'}
V1, V1N = {}, {}
for name, t in TIERS.items():
    V1[name] = P.load_prompt(t, 'v1', train_df)
    assert V1[name].hash(SETTINGS) in REG, f'{name}: prompt v1 khác bản đã khóa'
    V1N[name] = P.write_neutral(train_df, t)
    same_ex = [e['review_id'] for e in V1[name].exemplars] == [e['review_id'] for e in V1N[name].exemplars]
    print(f"{name}: v1n hash {V1N[name].hash(SETTINGS)[:16]}… | ví dụ giữ nguyên: {same_ex} | "
          f"ví dụ có chữ 'shopee': {sum('shopee' in e['text'].lower() for e in V1N[name].exemplars)}/{len(V1N[name].exemplars)}")
print('\nCâu mở đầu v1 :', V1['tier1'].system.split('.')[1].strip()[:120])
print('Câu mở đầu v1n:', V1N['tier1'].system.split('.')[1].strip()[:120])

tier1: v1n hash c1ef3bfc6bbb2a55… | ví dụ giữ nguyên: True | ví dụ có chữ 'shopee': 2/8
tier2: v1n hash 36ccc00e9c7664be… | ví dụ giữ nguyên: True | ví dụ có chữ 'shopee': 2/9
flat: v1n hash 354b64cbf1647ab3… | ví dụ giữ nguyên: True | ví dụ có chữ 'shopee': 3/14

Câu mở đầu v1 : Bạn đọc review tiếng Việt của người dùng ứng dụng thương mại điện tử Shopee trên Google Play và quyết định review đó có 
Câu mở đầu v1n: Bạn đọc review tiếng Việt của người dùng ứng dụng thương mại điện tử trên Google Play và quyết định review đó có ACTIONA


## Chạy thử 20 review Lazada

**Mẫu**

20 review Lazada text-only lấy bằng `random_state=42`, gọi cả tầng 1, tầng 2 và phẳng với bản v1n. Không đọc nhãn; chỉ đếm số review
mà v1n và v1 cho cùng nhãn, và ngoại suy chi phí.

In [4]:
pilot = TEXT['lazada'].sample(20, random_state=42)
PILOT_LOG = os.path.join(RES, 'paper_neutral_pilot_calls.jsonl')
PR = {}
for name, t in TIERS.items():
    PR[name] = results_from_log(PILOT_LOG, t) if os.path.exists(PILOT_LOG) and (results_from_log(PILOT_LOG, t).shape[0] == 20) \
        else classify(pilot, V1N[name], client, SETTINGS, PILOT_LOG, progress_every=0)
ref = {'tier1': pd.read_csv(os.path.join(RES, 'gd10_llm_lazada_tier1.csv')), 'tier2': pd.read_csv(os.path.join(RES, 'gd10_llm_lazada_tier2.csv')),
       'flat': pd.read_csv(os.path.join(RES, 'paper_flat_llm_lazada.csv'))}
for name in TIERS:
    m = PR[name][['review_id', 'pred_label']].merge(ref[name][['review_id', 'pred_label']], on='review_id', suffixes=(' v1n', ' v1'))
    print(f"{name}: cùng nhãn với v1 {int((m['pred_label v1n'] == m['pred_label v1']).sum())}/{len(m)} | fallback {int(PR[name]['fallback'].sum())}")
PC = pd.DataFrame([cost_summary(PR[n], PRICES, n) for n in TIERS])
print(PC[['bước', 'n_review', 'token_vào', 'token_ra', 'chi_phí_usd', 'usd_/1000_review']].round(5).to_string(index=False))
per = PC.set_index('bước')['usd_/1000_review'] / 1000
print(f"Ước lượng (giới hạn trên): Lazada 500 ≈ ${500 * per.sum():.3f} | Hold-out 1.000 ≈ ${1000 * per.sum():.3f}")

tier1: cùng nhãn với v1 20/20 | fallback 0
tier2: cùng nhãn với v1 12/12 | fallback 0
flat: cùng nhãn với v1 20/20 | fallback 0
 bước  n_review  token_vào  token_ra  chi_phí_usd  usd_/1000_review
tier1        20      22783       133      0.00350           0.17486
tier2        20      21083       100      0.00322           0.16112
 flat        20      35303       100      0.00536           0.26777
Ước lượng (giới hạn trên): Lazada 500 ≈ $0.302 | Hold-out 1.000 ≈ $0.604


## Chạy đầy đủ

**Điều kiện**

Chạy khi cờ tương ứng bật. Ba prompt v1n được khóa vào registry trước; mỗi tập gọi tầng 1, tầng 2 và phẳng trên mọi review, nối tiếp
log nếu đã chạy dở. Kết quả lưu `paper_neutral_{tập}_{tier1, tier2, flat}.csv`.

In [5]:
def run_set(s):
    for name, t in TIERS.items():
        log = os.path.join(RES, f'paper_neutral_{s}_{name}_calls.jsonl')
        done = results_from_log(log, t) if os.path.exists(log) else pd.DataFrame(columns=['review_id'])
        todo = TEXT[s][~TEXT[s]['review_id'].isin(done['review_id'])]
        new = classify(todo, V1N[name], client, SETTINGS, log, progress_every=250) if len(todo) else pd.DataFrame()
        out = TEXT[s][['review_id']].merge(pd.concat([done, new], ignore_index=True) if len(new) else done, on='review_id', how='left')
        assert out['pred_label'].notna().all()
        out.to_csv(os.path.join(RES, f'paper_neutral_{s}_{name}.csv'), index=False, encoding='utf-8-sig')
        cs = cost_summary(out, PRICES, f'{s} – {name}')
        print(f"{s} {name}: {len(out)} review | fallback {int(out['fallback'].sum())} | ${cs['chi_phí_usd']:.4f}")

if RUN_LAZADA or RUN_HOLDOUT:
    for name in TIERS:
        h, status = P.register(V1N[name], SETTINGS, RUN_DATE)
        print(f'{name} v1n: {h[:16]}… ({status})')
for s, flag in (('lazada', RUN_LAZADA), ('holdout', RUN_HOLDOUT)):
    if flag:
        run_set(s)
    else:
        print(f'{s}: chưa chạy (cờ tắt).')

tier1 v1n: c1ef3bfc6bbb2a55… (khớp bản đã khóa)
tier2 v1n: 36ccc00e9c7664be… (khớp bản đã khóa)
flat v1n: 354b64cbf1647ab3… (khớp bản đã khóa)


lazada tier1: 500 review | fallback 0 | $0.0872


lazada tier2: 500 review | fallback 0 | $0.0803


[tierflat] 250/287


lazada flat: 500 review | fallback 0 | $0.1336
holdout: chưa chạy (cờ tắt).


## Chấm điểm

**Điều kiện**

Chỉ chấm tập đã có đủ ba file dự đoán và khi ba prompt v1n đã nằm trong registry. Đây là lần đọc nhãn đã pre-register ở v3.6.

**So sánh**

Cascade LLM và flat LLM của v1n đặt cạnh v1 trên cùng review: Macro-F1 bốn lớp, chênh ghép cặp với bootstrap 95% CI và McNemar
(Holm trong họ); tầng 1 (F1, recall, precision ACTIONABLE); tầng 2 trên gold-actionable có dự đoán của cả hai bản; phân rã lỗi.
Kết quả v1 phải trùng số đã báo ở notebook 17.

In [6]:
LBL = {l: i for i, l in enumerate(FOUR_CLASSES)}
GOLD = {'lazada': 'lazada_external_test.csv', 'holdout': 'shopee_holdout.csv'}
V1_FILES = {'lazada': ('gd10_llm_lazada_tier1.csv', 'gd10_llm_lazada_tier2.csv', 'paper_flat_llm_lazada.csv'),
            'holdout': ('gd10_llm_holdout_tier1.csv', 'gd10_llm_holdout_tier2.csv', 'paper_flat_llm_holdout.csv')}
REF22 = {'lazada': 'paper_step2_e2e_lazada.csv', 'holdout': 'paper_step2_e2e_holdout.csv'}
REG = open(P.REGISTRY, encoding='utf-8').read()
LOCKED = all(V1N[n].hash(SETTINGS) in REG for n in TIERS)
OUT = {}
for s in ('lazada', 'holdout'):
    files = [os.path.join(RES, f'paper_neutral_{s}_{n}.csv') for n in TIERS]
    if not (LOCKED and all(os.path.exists(f) for f in files)):
        print(f'{s}: chưa chấm (chưa đủ dự đoán hoặc prompt v1n chưa khóa).'); continue
    gold = pd.read_csv(os.path.join(DATA, GOLD[s]), usecols=['review_id', 'gold_label'], encoding='utf-8-sig')
    ids, y = gold['review_id'].tolist(), gold['gold_label'].map(LBL).to_numpy()
    rd = lambda f: pd.read_csv(os.path.join(RES, f))
    c1, t1a, t2a = PA.llm_variant(ids, *(rd(f) for f in V1_FILES[s]), 'v1')
    c2, t1b, t2b = PA.llm_variant(ids, *(rd(f) for f in files), 'v1n')
    CFG = {**c1, **c2}
    tab, diff, yy, canon = PA.e2e_scores(y, CFG)
    ref = pd.read_csv(os.path.join(RES, REF22[s])).set_index('cấu hình')['Macro-F1 (4 lớp)'].astype(str)
    assert str(tab.set_index('cấu hình').loc['Cascade LLM – v1', 'Macro-F1 (4 lớp)']) == ref['Cascade LLM few-shot'], 'v1 cascade lệch notebook 17'
    assert str(tab.set_index('cấu hình').loc['Phẳng LLM – v1', 'Macro-F1 (4 lớp)']) == ref['Phẳng LLM few-shot'], 'v1 phẳng lệch notebook 17'
    mc = PA.mcnemar_table(yy, canon, f'end-to-end – {s}')
    dif = PA.variant_differences(diff, mc, [('Cascade LLM – v1n', 'Cascade LLM – v1'), ('Phẳng LLM – v1n', 'Phẳng LLM – v1')])
    t1 = PA.tier1_scores(y, {'tầng 1 – v1': {'llm': t1a}, 'tầng 1 – v1n': {'llm': t1b}})[0]
    both = (t2a >= 0) & (t2b >= 0)
    t2 = PA.tier2_scores(y, {'tầng 2 – v1': {'llm': t2a}, 'tầng 2 – v1n': {'llm': t2b}}, both)[0]
    act = PA.actionable_scores(y, CFG)
    err = PA.error_decomposition(y, CFG)
    view = tab.merge(act, on='cấu hình')[['cấu hình', 'Macro-F1 (4 lớp)', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'F1 OTHER', 'Recall ACT', 'Precision ACT']]
    view[['Recall ACT', 'Precision ACT']] = (view[['Recall ACT', 'Precision ACT']] * 100).round(1)
    print(f'===== {s} (n = {len(y)}; %)'); print(view.to_string(index=False))
    print('\nChênh v1n − v1 (lần chạy duy nhất):'); print(dif.round(3).to_string(index=False))
    print('\nTầng 1:'); print(t1[['chiến lược', 'F1 ACT', 'Recall ACT', 'Precision ACT', 'Macro-F1 (2 lớp)', 'f1 CI']].to_string(index=False))
    print(f'\nTầng 2 (gold-actionable có dự đoán của cả hai bản, n = {int(t2["n"].iloc[0])}):'); print(t2[['chiến lược', 'Macro-F1', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'Macro-F1 CI']].to_string(index=False))
    print('\nPhân rã lỗi:'); print(err.to_string(index=False))
    OUT[s] = {'e2e': view, 'diff': dif, 'tier1': t1, 'tier2': t2, 'errors': err, 'mcnemar': mc}
    for k, v in OUT[s].items():
        v.to_csv(os.path.join(RES, f'paper_neutral_{s}_{k}_scores.csv'), index=False, encoding='utf-8-sig')
if OUT:
    json.dump({'run_date': RUN_DATE, 'commit': COMMIT, 'amendment': 'v3.6', 'scored_sets': list(OUT),
               'v1n_hashes': {n: V1N[n].hash(SETTINGS) for n in TIERS}}, open(os.path.join(RES, 'paper_neutral_run_info.json'), 'w', encoding='utf-8'),
              ensure_ascii=False, indent=1)

===== lazada (n = 500; %)
         cấu hình Macro-F1 (4 lớp) F1 BUG F1 FEATURE F1 UX F1 OTHER  Recall ACT  Precision ACT
 Cascade LLM – v1             60.5   48.1       37.5  66.7     89.8        79.0           50.0
   Phẳng LLM – v1             59.5   45.0       34.5  70.8     87.6        81.5           44.9
Cascade LLM – v1n             60.5   49.0       34.3  70.6     88.1        87.7           46.7
  Phẳng LLM – v1n             61.5   48.7       40.0  69.4     87.8        87.7           46.1

Chênh v1n − v1 (lần chạy duy nhất):
                             so sánh  chênh Macro-F1  CI 2,5%  CI 97,5%  CI chứa 0  McNemar p (Holm)
Cascade LLM – v1n − Cascade LLM – v1           0.000   -4.885     4.865       True             0.378
    Phẳng LLM – v1n − Phẳng LLM – v1           2.009   -0.441     5.480       True             1.000

Tầng 1:
  chiến lược F1 ACT Recall ACT Precision ACT Macro-F1 (2 lớp)     f1 CI
 tầng 1 – v1   61.2       79.0          50.0             75.5 53.0–68.6
tầng 1

## Nhận xét

**Kết quả chính**

Bỏ tên Shopee khỏi prompt không làm thay đổi kết quả trên Lazada một cách có ý nghĩa. Cascade LLM giữ nguyên Macro-F1 60,5 (chênh 0,0,
khoảng tin cậy −4,9 đến +4,9); flat LLM tăng từ 59,5 lên 61,5 (chênh +2,0, khoảng tin cậy −0,4 đến +5,5, McNemar p Holm 1,000).
Mức giảm khoảng 20 điểm so với Hold-out vì vậy không giải thích được bằng tên ứng dụng trong prompt.

**Nơi mất điểm**

Trên Lazada, tầng 2 của LLM vẫn tốt (Macro-F1 77,9 với v1 và 77,7 với v1n trên 81 review gold-actionable). Phần lớn lỗi nằm ở ranh
giới ACTIONABLE / OTHER: precision ACTIONABLE của tầng 1 chỉ 50,0 (v1) và 46,7 (v1n), và giữ nhầm OTHER chiếm 69,6–80,6% số lỗi end-to-end.
Bản v1n gọi ACTIONABLE rộng tay hơn (recall 79,0 → 87,7, precision 50,0 → 46,7) nên F1 ACTIONABLE gần như không đổi (61,2 và 60,9).

**Hệ quả**

Khác biệt nằm ở nội dung review của hai ứng dụng, cụ thể là những than phiền mà LLM coi là việc của đội phát triển app trong khi
guideline xếp vào OTHER. Câu hỏi này cần được xem trên các review bị giữ nhầm, không phải bằng cách chỉnh tiếp prompt trên Lazada.

**Chi phí và tái lập**

Ba prompt v1n được khóa vào registry trước khi chạy; 1.500 lần gọi trên Lazada không có lần thử lại, fallback hay lỗi API, tổng khoảng
0,30 USD (thêm 0,01 USD chạy thử). Hai lần thực thi đầu bị dừng vì vượt giới hạn 5.400 giây của một cell khi API phản hồi chậm; mỗi lần
chạy lại đều nối tiếp log, và mỗi prompt có đúng 500 lần gọi cho 500 review khác nhau, không có lần gọi trùng.